# Clase 203 — Reentrenamiento programado (Prefect)

Pipeline `pull → validate → train → evaluate → champion-challenger` con Prefect (más liviano que Airflow para una clase).

Requiere: `pip install prefect mlflow scikit-learn`. Para correr en cloud: `prefect cloud login`.

## Setup

In [ ]:
import os, tempfile, shutil
from pathlib import Path
WORK = Path(tempfile.gettempdir()) / 'retrain_demo'
if WORK.exists(): shutil.rmtree(WORK)
WORK.mkdir(); os.chdir(WORK)

import mlflow
mlflow.set_tracking_uri(f'file:{WORK}/mlruns')
mlflow.set_experiment('continual-training')

## 1. Tareas individuales (idempotentes)

In [ ]:
from prefect import flow, task, get_run_logger
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
import joblib, json, numpy as np, hashlib

@task(retries=2, retry_delay_seconds=10)
def pull_data(execution_date: str):
    log = get_run_logger()
    X, y = load_iris(return_X_y=True)
    # Simular drift incremental: inyectar ruido proporcional al día
    seed = int(hashlib.md5(execution_date.encode()).hexdigest(), 16) % 1000
    rng = np.random.default_rng(seed)
    X = X + rng.normal(0, 0.1, X.shape)
    log.info(f'pulled {len(X)} rows for {execution_date}')
    return X, y

@task
def validate_data(X, y):
    log = get_run_logger()
    assert not np.isnan(X).any(), 'NaN in features'
    assert set(np.unique(y)) == {0, 1, 2}, 'unexpected labels'
    log.info('data validation passed')

@task
def train(X, y, execution_date: str):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42)
    with mlflow.start_run(run_name=f'train_{execution_date}') as run:
        m = RandomForestClassifier(n_estimators=100, random_state=42).fit(Xtr, ytr)
        f1 = f1_score(yte, m.predict(Xte), average='macro')
        mlflow.log_metric('f1', f1)
        mlflow.log_param('execution_date', execution_date)
        mlflow.sklearn.log_model(m, name='model')
        return {'run_id': run.info.run_id, 'f1': f1}

@task
def promote_if_better(challenger: dict, alias='champion', delta=0.005):
    log = get_run_logger()
    client = mlflow.tracking.MlflowClient()
    # Buscar champion actual
    try:
        champ_mv = client.get_model_version_by_alias('iris-prod', alias)
        champ_run = client.get_run(champ_mv.run_id)
        champ_f1 = champ_run.data.metrics.get('f1', 0)
    except Exception:
        champ_f1 = 0
        log.info('no champion yet — promoting first')

    if challenger['f1'] > champ_f1 + delta:
        mv = mlflow.register_model(f"runs:/{challenger['run_id']}/model", 'iris-prod')
        client.set_registered_model_alias('iris-prod', alias, mv.version)
        log.info(f'PROMOTED v{mv.version} (f1 {challenger["f1"]:.4f} > champion {champ_f1:.4f})')
        return {'promoted': True, 'version': mv.version}
    log.info(f'SKIPPED (challenger {challenger["f1"]:.4f} not > champion {champ_f1:.4f} + {delta})')
    return {'promoted': False}

## 2. Flow (el DAG)

In [ ]:
@flow(name='daily-retrain')
def daily_retrain(execution_date: str):
    X, y = pull_data(execution_date)
    validate_data(X, y)
    challenger = train(X, y, execution_date)
    return promote_if_better(challenger)

# Correr 5 días simulados
for day in ['2026-06-01', '2026-06-02', '2026-06-03', '2026-06-04', '2026-06-05']:
    print(f'\n=== {day} ===')
    result = daily_retrain(day)
    print(result)

## 3. Estado final del Model Registry

In [ ]:
client = mlflow.tracking.MlflowClient()
for mv in client.search_model_versions("name='iris-prod'"):
    aliases = ','.join(mv.aliases) if mv.aliases else '-'
    print(f'v{mv.version}: aliases={aliases}, run={mv.run_id[:8]}')

champ = client.get_model_version_by_alias('iris-prod', 'champion')
print(f'\nCHAMPION: v{champ.version}')

## 4. Schedule + deploy (Prefect Cloud)

In [ ]:
deploy_script = '''\
# deploy.py — corré con: python deploy.py
from prefect import serve
from prefect.client.schemas.schedules import CronSchedule
from your_flow_module import daily_retrain

if __name__ == "__main__":
    deployment = daily_retrain.to_deployment(
        name="daily-retrain-prod",
        schedule=CronSchedule(cron="0 2 * * *", timezone="UTC"),  # 02:00 diario
        parameters={"execution_date": "{{ run.scheduled_start_time | date }}"},
        tags=["prod", "ml-retrain"],
    )
    serve(deployment)
'''
print(deploy_script)

## Ejercicio guiado

1. Agregá una tarea `check_drift` al inicio que computa PSI vs un baseline. Si <0.1: `return` temprano (skip).
2. Convertí `promote_if_better` a `set alias '@challenger'` en vez de `@champion` — la promoción a champion la decidís manualmente o tras shadow OK (Clase 204).
3. Agregá `on_failure_callback` que postea a un Slack webhook (stub si no tenés webhook real).
4. Implementá backfill: corré el flow para `[2026-06-01, ..., 2026-06-07]` y verificá idempotencia.

## Conclusiones

- DAG + champion-challenger = retraining sin regresiones.
- Idempotencia es responsabilidad de cada tarea (insert con upsert, sobrescribir destinos).
- Schedule fijo + trigger por drift se complementan; uno solo deja gaps.
- Promoción auto solo si el delta justifica + shadow OK (Clase 204).

## ✅ Soluciones de los ejercicios

Soluciones de los 5 ejercicios del README. Airflow/Prefect y MLflow no están instalados, así que mostramos el **DAG real** como se escribiría, y ejecutamos la *lógica* de cada tarea con Python puro: el pipeline `pull→validate→train→evaluate→promote`, la regla champion-challenger, la idempotencia por `execution_date`, el gate por drift y el backfill. Todo el valor de un scheduler de ML está en esa lógica, no en el runner.

In [ ]:
import numpy as np, pandas as pd
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
print('base para el pipeline de reentrenamiento lista.')

### Ejercicio 1 — DAG mínimo de 5 tareas

Cada tarea es una función Python; el DAG las encadena `pull_data → validate_data → train → evaluate → promote`, con schedule diario. Mostramos el DAG de Airflow y ejecutamos las 5 funciones con un runner secuencial.

In [ ]:
airflow_dag = '''
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime
with DAG("retrain", schedule="@daily", start_date=datetime(2024,1,1), catchup=False) as dag:
    t1 = PythonOperator(task_id="pull_data", python_callable=pull_data)
    t2 = PythonOperator(task_id="validate_data", python_callable=validate_data)
    t3 = PythonOperator(task_id="train", python_callable=train)
    t4 = PythonOperator(task_id="evaluate", python_callable=evaluate)
    t5 = PythonOperator(task_id="promote", python_callable=promote)
    t1 >> t2 >> t3 >> t4 >> t5
'''
def pull_data(seed=0):
    X, y = make_classification(n_samples=1000, n_features=8, random_state=seed)
    return {'X': X, 'y': y}
def validate_data(ctx):
    assert not np.isnan(ctx['X']).any(), 'datos con NaN'
    return ctx
def train(ctx):
    Xtr, Xte, ytr, yte = train_test_split(ctx['X'], ctx['y'], test_size=0.3, random_state=0)
    model = LogisticRegression(max_iter=500).fit(Xtr, ytr)
    return {'model': model, 'Xte': Xte, 'yte': yte}
def evaluate(ctx):
    ctx['f1'] = f1_score(ctx['yte'], ctx['model'].predict(ctx['Xte']))
    return ctx
def promote(ctx):
    return {'promoted': True, 'f1': round(ctx['f1'], 4)}

# runner secuencial (lo que hace el scheduler)
result = promote(evaluate(train(validate_data(pull_data(0)))))
print('DAG ejecutado ->', result)
assert result['promoted'] and 0 <= result['f1'] <= 1
print('OK — las 5 tareas corrieron en orden (Airflow/Prefect solo agrega scheduling + retries + UI).')

### Ejercicio 2 — Champion-challenger

En `promote`, el challenger reemplaza al champion **solo si mejora por un margen** (`f1 > champion.f1 + 0.005`); si no, se loguea `[skipped]` y el champion queda intacto.

In [ ]:
def promote_if_better(champion_f1, challenger_f1, margin=0.005):
    if challenger_f1 > champion_f1 + margin:
        return {'action': 'promote', 'new_champion_f1': challenger_f1}
    return {'action': 'skipped', 'reason': 'challenger no es significativamente mejor',
            'champion_f1': champion_f1}

assert promote_if_better(0.80, 0.81)['action'] == 'promote'      # +0.01 > margen
assert promote_if_better(0.80, 0.802)['action'] == 'skipped'     # +0.002 < margen
assert promote_if_better(0.80, 0.79)['action'] == 'skipped'      # peor
print(promote_if_better(0.80, 0.802))
print('OK — el margen evita promover por ruido estadístico.')

### Ejercicio 3 — Idempotencia por `execution_date`

Correr el DAG dos veces para la misma fecha NO debe duplicar filas ni registros. Se logra con un *upsert* por clave `execution_date` (no un append ciego).

In [ ]:
processed = {}   # simula data/processed indexado por execution_date
runs = []        # simula registros de MLflow

def dag_run(execution_date, rows):
    processed[execution_date] = rows                 # upsert: sobrescribe, no duplica
    if not any(r['date'] == execution_date for r in runs):
        runs.append({'date': execution_date, 'f1': 0.83})

dag_run('2024-01-15', rows=100)
dag_run('2024-01-15', rows=100)   # re-ejecución de la MISMA fecha
assert len(processed) == 1 and processed['2024-01-15'] == 100
assert len([r for r in runs if r['date'] == '2024-01-15']) == 1
print('processed dates:', list(processed), '| runs para esa fecha:', len(runs))
print('OK — idempotente: la re-ejecución de un día no ensucia downstream.')

### Ejercicio 4 — Trigger por drift

Una tarea inicial `check_drift` corta el DAG si NO hay drift (`raise AirflowSkipException`) → solo se reentrena cuando hace falta.

In [ ]:
class SkipException(Exception):    # imita airflow.exceptions.AirflowSkipException
    pass

def psi_quick(ref, cur, bins=10):
    edges = np.quantile(ref, np.linspace(0, 1, bins + 1)); edges[0], edges[-1] = -np.inf, np.inf
    pr, _ = np.histogram(ref, bins=edges); pc, _ = np.histogram(cur, bins=edges)
    pr = np.clip(pr / pr.sum(), 1e-6, 1); pc = np.clip(pc / pc.sum(), 1e-6, 1)
    return float(np.sum((pc - pr) * np.log(pc / pr)))

def check_drift(ref, cur, threshold=0.2):
    p = psi_quick(ref, cur)
    if p <= threshold:
        raise SkipException(f'PSI={p:.3f} <= {threshold}: sin drift, skip retrain')
    return f'PSI={p:.3f} > {threshold}: continúa el DAG'

rng = np.random.default_rng(0)
ref = rng.normal(0, 1, 3000)
# caso sin drift -> skip
try:
    check_drift(ref, rng.normal(0, 1, 3000)); raise SystemExit('debió skipear')
except SkipException as e:
    print('sin drift ->', e)
# caso con drift -> continúa
print('con drift ->', check_drift(ref, rng.normal(1.5, 1, 3000)))
print('OK — el DAG entrena solo cuando el drift lo justifica (ahorra cómputo).')

### Ejercicio 5 — Backfill sin duplicar downstream

Tras fixear un bug en `validate_data`, `airflow dags backfill --start-date X --end-date Y` re-procesa los días afectados. Gracias a la idempotencia del ej.3, no se duplican registros.

In [ ]:
def backfill(start_day, end_day):
    reprocessed = []
    for d in range(start_day, end_day + 1):
        date = f'2024-01-{d:02d}'
        dag_run(date, rows=100)          # mismo upsert idempotente
        reprocessed.append(date)
    return reprocessed

before = dict(processed)
done = backfill(10, 16)                  # re-procesa una semana
backfill(10, 16)                         # correrlo DOS veces no duplica
assert all(processed[d] == 100 for d in done)
assert len(processed) == len(set(processed)), 'no debe haber claves duplicadas'
print('backfilled:', done)
print('OK — backfill idempotente: re-procesa sin generar filas/registros duplicados.')